# 02 - Metadata Processing & Label Construction

This notebook extracts sample-level metadata from the GEO series matrix files for GSE53482 and prepares the corresponding labels for supervised machine learning.

The expression matrices generated in Notebook 01 contain sample identifiers but do not contain the final classification labels required by the machine-learning workflow. This notebook therefore extracts and parses the GEO sample metadata, constructs a binary PMF-versus-control outcome, and aligns the metadata with the corresponding expression matrices.

The gene-expression and miRNA-expression datasets are processed independently, while retaining the corresponding sample identifiers and disease labels for subsequent model comparison.

### Main steps

* Load the cleaned gene-expression and miRNA-expression matrices
* Extract sample metadata from the original GEO files
* Parse metadata into structured tables
* Inspect disease-group composition
* Construct binary PMF/control labels
* Examine potential biological and technical confounding
* Align expression samples with their corresponding metadata
* Validate sample and label alignment
* Save aligned expression matrices and metadata for model training


### Import Required Libraries

In [1]:
# Load required libraries
import pandas as pd
import sys
import os

# Add project root to system path for module imports
sys.path.append(os.path.abspath(".."))

# Import custom functions for metadata extraction and parsing
from Src.Data_Preprocessing import extract_metadata, parse_metadata

### Define Raw Data File Paths

Specify file paths for both the raw GEP and miEP GEO series matrix files for metadata extraction and downstream processing

In [2]:
# File path for raw gene expression GEO series matrix
gene_file = "../Data/Raw/GSE53482-GPL13667_series_matrix.txt.gz"

# File path for raw miRNA expression GEO series matrix
mirna_file = "../Data/Raw/GSE53482-GPL14613_series_matrix.txt.gz"

### Load Processed Expression Datasets

Load cleaned gene expression and miRNA datasets for alignment with metadata.

In [3]:
# Load processed gene expression dataset
gene_df = pd.read_csv("../Data/Processed/gene_expression.csv", index_col=0)

# Load processed miRNA expression dataset
mirna_df = pd.read_csv("../Data/Processed/mirna_expression.csv", index_col=0)

### Extract and Parse Gene Expression Metadata

In [4]:
# Extract raw metadata from gene expression file
meta_gene_raw = extract_metadata(gene_file)

# Parse and clean metadata into structured format
meta_gene_clean = parse_metadata(meta_gene_raw)

# Preview cleaned metadata
meta_gene_clean.head()

,supplier,cell type,disease,jak2v617f,tissue
GSM1294472,Vannucchi,MPD,PMF,neg,PB
GSM1294473,Vannucchi,MPD,PMF,pos,PB
GSM1294474,Vannucchi,MPD,PMF,pos,PB
GSM1294475,Vannucchi,MPD,PMF,pos,PB
GSM1294476,Vannucchi,MPD,PMF,pos,PB


### Inspect Disease Distribution (Gene)

Examine the distribution of disease categories in the gene metadata to validate class composition before label encoding.

In [5]:
# Check distribution of disease categories
meta_gene_clean["disease"].value_counts()

disease
PMF       42
PB CTR    16
BM CTR    15
Name: count, dtype: int64

### Create Binary Labels for Classification (Gene)

Convert disease categories into binary labels for supervised learning (PMF vs Control).

In [6]:
# Map disease categories to binary labels
meta_gene_clean["label"] = meta_gene_clean["disease"].map({
    "PMF": 1,
    "PB CTR": 0,
    "BM CTR": 0
})

# Verify label distribution
meta_gene_clean["label"].value_counts()

label
1    42
0    31
Name: count, dtype: int64

### Check for Potential Biological and Technical Confounding

Sample-level characteristics are examined for strong associations with the PMF/control label.

Differences in tissue, cell type, supplier, or disease-associated molecular characteristics may be correlated with disease status. A classifier may therefore capture biological or technical differences that are associated with the study groups rather than disease-specific expression changes alone.

The observed metadata distributions are treated as potential sources of confounding rather than as evidence that the machine-learning models necessarily learned these variables directly.

These checks are diagnostic only. No samples are removed or modified at this stage.

In [7]:
# Examine available metadata variables against the binary classification label.
# Strongly imbalanced distributions may indicate potential confounding.

confounding_variables = [
    "supplier",
    "cell type",
    "jak2v617f",
    "tissue"
]

for column in confounding_variables:
    print(f"\n--- {column} ---")
    print(pd.crosstab(
        meta_gene_clean[column],
        meta_gene_clean["label"]
    ))


--- supplier ---
label       0   1
supplier         
Barosi      0  10
Cazzola     1   0
Rambaldi    0   4
Vannucchi  30  28

--- cell type ---
label       0   1
cell type        
CTR        31   0
MPD         0  42

--- jak2v617f ---
label       0   1
jak2v617f        
neg        31  19
pos         0  23

--- tissue ---
label    0   1
tissue        
BM      15   0
PB      16  42


### Align Gene Expression Data with Metadata

Ensure that gene expression samples are aligned with the metadata by matching sample identifiers and verifying consistency.

In [8]:
# Align gene expression data to metadata sample order
gene_df = gene_df.loc[meta_gene_clean.index]

# Verify alignment between gene expression and metadata
assert gene_df.index.equals(meta_gene_clean.index)

### Save Processed and Aligned Datasets

Save the cleaned and aligned gene expression data and corresponding metadata for use in downstream analysis and modelling.

In [9]:
# Save aligned gene expression dataset
gene_df.to_csv("../Data/Processed/gene_expression_aligned.csv")

# Save cleaned and labeled metadata
meta_gene_clean.to_csv("../Data/Processed/gene_metadata.csv")

### Extract and Parse miRNA Metadata

Extract metadata from the miRNA GEO series matrix file and parse it into a structured format for downstream analysis.

In [10]:
# Extract raw metadata from miRNA file
meta_mirna_raw = extract_metadata(mirna_file)

# Parse and clean metadata into structured format
meta_mirna_clean = parse_metadata(meta_mirna_raw)

# Preview cleaned metadata
meta_mirna_clean.head()

,supplier,cell type,disease,jak2v617f,tissue
GSM1294399,Vannucchi,MPD,PMF,pos,PB
GSM1294400,Vannucchi,MPD,PMF,neg,PB
GSM1294401,Vannucchi,MPD,PMF,pos,PB
GSM1294402,Vannucchi,MPD,PMF,pos,PB
GSM1294403,Vannucchi,MPD,PMF,neg,PB


### Inspect Disease Distribution (miRNA)

Examine the distribution of disease categories in the miRNA metadata to validate class composition before label encoding.

In [11]:
# Check distribution of disease categories
meta_mirna_clean["disease"].value_counts()

disease
PMF       42
PB CTR    16
BM CTR    15
Name: count, dtype: int64

### Create Binary Labels for Classification (miRNA)

Convert disease categories into binary labels for supervised learning (PMF vs Control).

In [12]:
# Map disease categories to binary labels
meta_mirna_clean["label"] = meta_mirna_clean["disease"].map({
    "PMF": 1,
    "PB CTR": 0,
    "BM CTR": 0
})

# Verify label distribution
meta_mirna_clean["label"].value_counts()

label
1    42
0    31
Name: count, dtype: int64

### Check for Potential Biological and Technical Confounding

The same metadata checks are performed for the miRNA samples to determine whether sample characteristics are distributed differently between PMF and control groups.

These checks are diagnostic and do not modify the dataset.

In [13]:
# Examine available metadata variables against the binary classification label.
# This checks whether potential confounders are unevenly distributed between
# PMF and control samples.

for column in confounding_variables:
    print(f"\n--- {column} ---")
    print(pd.crosstab(
        meta_mirna_clean[column],
        meta_mirna_clean["label"]
    ))


--- supplier ---
label       0   1
supplier         
Barosi      0  10
Cazzola     1   0
Rambaldi    0   4
Vannucchi  30  28

--- cell type ---
label       0   1
cell type        
CTR        31   0
MPD         0  42

--- jak2v617f ---
label       0   1
jak2v617f        
neg        31  19
pos         0  23

--- tissue ---
label    0   1
tissue        
BM      15   0
PB      16  42


### Align miRNA Expression Data with Metadata

Ensure that miRNA expression samples are aligned with the metadata by matching sample identifiers and verifying consistency.

In [14]:
# Align miRNA expression data to metadata sample order
mirna_df = mirna_df.loc[meta_mirna_clean.index]

# Verify alignment between miRNA expression and metadata
assert mirna_df.index.equals(meta_mirna_clean.index)

### Save Processed and Aligned miRNA Datasets

Save the cleaned and aligned miRNA expression data and corresponding metadata for downstream analysis and modelling.

In [15]:
# Save aligned miRNA expression dataset
mirna_df.to_csv("../Data/Processed/mirna_expression_aligned.csv")

# Save cleaned and labeled metadata
meta_mirna_clean.to_csv("../Data/Processed/mirna_metadata.csv")

### Summary

Sample-level metadata were extracted from the GSE53482 GEO series matrix files and used to construct the binary classification outcome.

The resulting cohort contains:

* **42 PMF samples**
* **31 control samples**, comprising 16 PB CTR and 15 BM CTR samples
* **73 samples in total**

The gene-expression and miRNA-expression datasets were independently aligned to their corresponding metadata, and the resulting sample orders and disease labels were verified.

The metadata inspection also identified substantial associations between disease status and several sample characteristics, including cell type, tissue, JAK2 V617F status, and supplier. These characteristics therefore represent potential sources of biological or technical confounding in the classification task.

No samples were removed and no confounding adjustment was performed. The observed associations are retained as an important consideration when interpreting the predictive performance obtained in the subsequent machine-learning analyses.

The aligned expression matrices and metadata tables were saved for downstream model training and cross-validation.
